In [12]:
import os

from langchain.chat_models import init_chat_model

main_llm = init_chat_model(
    model="deepseek-chat",
    temperature=0.0,
    api_key=os.getenv("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai",
    max_tokens=20,
)

sum_llm = init_chat_model(
    model="deepseek-chat",
    temperature=0.0,
    api_key=os.getenv("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai",
    max_tokens=20,
)

from langchain.agents.middleware import SummarizationMiddleware

middleware = SummarizationMiddleware(
    model=sum_llm,  # 指定摘要模型
    max_token_before_summary=30,  # 最大token数量，阈值
    keep=("messages", 5),  # 保存最近 5 条数据
    summary_prompt="用20个字概括要点",
)

from langchain.agents import create_agent

agent = create_agent(
    model=main_llm, tools=[], system_prompt="只用一句极短的中文回答, 不超过30个字", middleware=[middleware]
)

conversation = ["RAG是什么?", "有什么作用？", "主要缺点?", "有什么优点", "一句话总结"]

state = {"messages": []}

from langchain.messages import HumanMessage

for i, question in enumerate(conversation, 1):
    user_msg = HumanMessage(content=question)
    state = await agent.ainvoke({"messages": state["messages"] + [user_msg]})
    answer = state["messages"][-1].content
    print(f"第{i}论会话: ")
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(state)

第1论会话: 
Q: RAG是什么?
A: RAG是检索增强生成，结合检索与生成模型提升回答准确性。
{'messages': [HumanMessage(content='RAG是什么?', additional_kwargs={}, response_metadata={}, id='77cd7389-8935-4037-9f39-0f61df98ed3c'), AIMessage(content='RAG是检索增强生成，结合检索与生成模型提升回答准确性。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 20, 'total_tokens': 36, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 20}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '9f28b08f-bb80-4abc-bc07-a32afa9a5999', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f2199-75a9-7ef3-8067-c7323e3ec59d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 16, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})